# Clase 3 (Parte 3): RAG Avanzado, PDFs y Multimodalidad
**Prof. Leticia Rodriguez | Universidad de Buenos Aires**

En este laboratorio final llevaremos nuestra arquitectura RAG a un nivel de producción realista. Exploraremos tres bloques fundamentales:
1. **Documentos Reales (PDFs):** Parseo y extracción de texto estructurado.
2. **Multimodal RAG:** Indexación visual y *grounding* utilizando imágenes nativas.
3. **RAG Avanzado:** Patrones de *Query Rewriting* (Pre-Retrieval) y *Re-ranking* (Post-Retrieval) para domar consultas ambiguas.

In [1]:
# Instalamos el stack completo de dependencias
!pip install -q google-generativeai pypdf faiss-cpu numpy pillow requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 29.3 MB/s eta 0:00:00


In [5]:
import os
import requests
import numpy as np
import faiss
from PIL import Image
from io import BytesIO
from pypdf import PdfReader
import google.genai as genai
from google.colab import userdata

# Configuración del cliente y selección de modelos directa
gemini_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_key)

MODELO_LLM = 'gemini-2.5-flash'
MODELO_EMBEDDING = 'gemini-embedding-2'

print("Entorno inicializado. Modelos configurados:")
print(f"- LLM: {MODELO_LLM}")
print(f"- Embeddings: {MODELO_EMBEDDING}")

Entorno inicializado. Modelos configurados:
- LLM: gemini-2.5-flash
- Embeddings: gemini-embedding-2


## SECCIÓN 1: Ingesta de Documentos y RAG con PDFs
Los sistemas corporativos viven de PDFs. Descargaremos un documento académico real, extraeremos su texto con `pypdf`, lo fragmentaremos y lo indexaremos en FAISS.

In [ ]:
# 1. Descargamos un PDF académico con texto real para evitar matrices vacías
url_pdf = "https://arxiv.org/pdf/1706.03762.pdf" # Attention Is All You Need
response = requests.get(url_pdf)
with open("paper.pdf", "wb") as f:
    f.write(response.content)

# 2. Extraemos el texto de las primeras 2 páginas usando pypdf
reader = PdfReader("paper.pdf")
texto_pdf = ""
for i in range(min(2, len(reader.pages))):
    texto_pdf += reader.pages[i].extract_text() + "\n"

# 3. Chunking manual básico (400 caracteres con verificación de longitud)
chunks_pdf = [texto_pdf[i:i+400] for i in range(0, len(texto_pdf), 300) if len(texto_pdf[i:i+400].strip()) > 50]

# 4. Generación de Embeddings directos e Indexación FAISS
vectores_pdf = []
for chunk in chunks_pdf:
    vector = client.models.embed_content(model=MODELO_EMBEDDING, contents=chunk).embeddings[0].values
    vectores_pdf.append(vector)

print("Longitud: ",len(vectores_pdf))

matriz_pdf = np.array(vectores_pdf).astype('float32')
faiss.normalize_L2(matriz_pdf)

index_pdf = faiss.IndexFlatIP(matriz_pdf.shape[1])
index_pdf.add(matriz_pdf)

print(f"Se procesaron {len(chunks_pdf)} chunks del PDF y se guardaron en la base vectorial.")

Longitud:  24
Se procesaron 24 chunks del PDF y se guardaron en la base vectorial.


In [ ]:
# 5. Consulta sobre el PDF indexado
query_pdf = "¿Cuál es la principal ventaja de la arquitectura Transformer mencionada en el documento?"
q_vec = np.array(client.models.embed_content(model=MODELO_EMBEDDING, contents=query_pdf).embeddings[0].values).astype('float32').reshape(1, -1)
faiss.normalize_L2(q_vec)

_, indices = index_pdf.search(q_vec, k=1)
contexto_pdf = chunks_pdf[indices[0][0]]

prompt_pdf = f"Responde basándote estrictamente en este texto extraído de un PDF:\n\n{contexto_pdf}\n\nPregunta: {query_pdf}"
respuesta_pdf = client.models.generate_content(contents=prompt_pdf, model=MODELO_LLM)

print("--- RESPUESTA BASADA EN EL PDF ---")
print(respuesta_pdf.text)

--- RESPUESTA BASADA EN EL PDF ---
Basándose estrictamente en el texto, la principal ventaja de la arquitectura Transformer mencionada es que es el primer modelo de transducción que se basa **completamente en la auto-atención (self-attention) para calcular representaciones de su entrada y salida sin utilizar RNNs (redes neuronales recurrentes) alineadas por secuencia o convolución.**


## SECCIÓN 2: Multimodal RAG (Imágenes + Texto)
Gemini procesa imágenes de forma nativa. Podemos indexar imágenes usando descripciones textuales y, una vez recuperadas mediante álgebra lineal, pasarle la imagen real a la LLM para un análisis visual profundo.

In [ ]:
def cargar_imagen_web(url):
    res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    return Image.open(BytesIO(res.content))

# Base de datos multimodal simulada (Catálogo de hardware)
catalogo = [
    {
        "id": "GPU NVIDIA",
        "desc": "Tarjeta gráfica NVIDIA moderna, con disipadores masivos, ideal para entrenamiento de LLMs.",
        # Usamos Unsplash: es un CDN de imágenes libre, súper estable y no rompe enlaces.
        "img": cargar_imagen_web("https://images.unsplash.com/photo-1591488320449-011701bb6704?w=400")
    },
    {
        "id": "Placa Arduino",
        "desc": "Microcontrolador Arduino Uno, placa azul con puertos de conexión para electrónica básica.",
        # Este de Wikipedia suele ser estable, pero si falla podés usar cualquier foto directa (.jpg)
        "img": cargar_imagen_web("https://i5.walmartimages.com/seo/Arduino-Uno-R3-SMD-with-CH340-Chipset-Clone-Includes-USB-Cable_6afc41ec-5ef9-435a-96c6-877d66e1976d_1.524aeba9aa11757c4a5fac69370e8ea5.jpeg?odnHeight=768&odnWidth=768&odnBg=FFFFFF")
    }
]

# Indexamos basándonos en la descripción textual
vectores_catalogo = [client.models.embed_content(model=MODELO_EMBEDDING, contents=item["desc"]).embeddings[0].values for item in catalogo]
matriz_cat = np.array(vectores_catalogo).astype('float32')
faiss.normalize_L2(matriz_cat)
index_catalogo = faiss.IndexFlatIP(matriz_cat.shape[1])
index_catalogo.add(matriz_cat)

# Consulta del usuario
query_hardware = "Necesito analizar los componentes visibles de una tarjeta para entrenar Inteligencia Artificial."
q_vec_hw = np.array(client.models.embed_content(model=MODELO_EMBEDDING, contents=query_hardware).embeddings[0].values).astype('float32').reshape(1, -1)
faiss.normalize_L2(q_vec_hw)

_, indices_hw = index_catalogo.search(q_vec_hw, k=1)
item_recuperado = catalogo[indices_hw[0][0]]

print(f"--- IMAGEN RECUPERADA DE LA BASE VECTORIAL: {item_recuperado['id']} ---")

# Prompt Multimodal: Pasamos Texto + Imagen al modelo
prompt_multimodal = [
    "Eres un experto en hardware. Basándote en la imagen adjunta, responde la consulta del usuario detallando lo que ves.",
    item_recuperado["img"], # Pasamos el objeto imagen directamente para visual grounding
    f"Consulta: {query_hardware}"
]

respuesta_multimodal = client.models.generate_content(contents=prompt_multimodal, model=MODELO_LLM)
print("\n--- ANÁLISIS DE LA LLM SOBRE LA IMAGEN RECUPERADA ---")
print(respuesta_multimodal.text)

--- IMAGEN RECUPERADA DE LA BASE VECTORIAL: GPU NVIDIA ---

--- ANÁLISIS DE LA LLM SOBRE LA IMAGEN RECUPERADA ---
¡Excelente! Como experto en hardware, me complace analizar las tarjetas gráficas visibles en la imagen para determinar su idoneidad y características relevantes para el entrenamiento de Inteligencia Artificial.

En la imagen, podemos observar dos tarjetas gráficas de alto rendimiento, ambas de la marca NVIDIA, que representan distintas generaciones y enfoques de diseño.

---

### Observaciones Generales:

Ambas tarjetas son de formato **doble slot** y de longitud completa, lo que indica que están diseñadas para sistemas con suficiente espacio interno y fuentes de alimentación robustas. Se conectan a la placa base a través de una interfaz **PCI Express x16**, el estándar para tarjetas gráficas de alto rendimiento. Ambas utilizan el ecosistema de computación paralela **CUDA** de NVIDIA, esencial para el entrenamiento de modelos de IA.

---

### Análisis Detallado de Cada Tarj

## SECCIÓN 3: Patrones de RAG Avanzado
Las consultas humanas suelen ser vagas. Usaremos **Query Rewriting** para que la LLM mejore la pregunta antes de buscar, y **Re-ranking** para ordenar los resultados de forma post-recuperación (ej: priorizando regulaciones vigentes).

In [6]:
# --- 3.1 QUERY REWRITING (Pre-Retrieval) ---
query_vaga = "che, cuantos faltas puedo tener en ciencia de datos para no quedar afuera?"

prompt_optimizador = f"""
Eres un traductor de consultas para bases vectoriales.
Reescribe la siguiente consulta informal de un estudiante en una pregunta técnica, formal y optimizada para buscar normativas académicas.
Devuelve SOLO la pregunta reescrita, sin introducciones.

Original: {query_vaga}
"""
query_optimizada = client.models.generate_content(contents=prompt_optimizador, model=MODELO_LLM).text.strip()

print("--- PRE-RETRIEVAL (QUERY REWRITING) ---")
print(f"Original: {query_vaga}")
print(f"Optimizada: {query_optimizada}")

--- PRE-RETRIEVAL (QUERY REWRITING) ---
Original: che, cuantos faltas puedo tener en ciencia de datos para no quedar afuera?
Optimizada: Recuperar normativas académicas universitarias que especifiquen el umbral máximo de inasistencias permitidas para la carrera o programa de estudios en Ciencia de Datos, detallando las consecuencias de exceder dicho límite, tales como la pérdida de regularidad, la desaprobación de la asignatura o la baja académica. Incluir disposiciones sobre justificación de ausencias.


In [7]:
# --- 3.2 RE-RANKING (Post-Retrieval) ---
# Simulamos resultados devueltos por FAISS con metadatos asociados
candidatos_faiss = [
    {"texto": "Reglamento 2022 (Histórico): Se exige un 75% de asistencia mínima para posgrados.", "vigente": False},
    {"texto": "Resolución UBA 2026: La asistencia obligatoria en la Maestría en Ciencias de Datos es del 80%.", "vigente": True},
    {"texto": "Reglamento General: Las llegadas tarde de más de 30 minutos cuentan como media falta.", "vigente": True}
]

print("\n--- POST-RETRIEVAL (RESULTADOS CRUDOS DE FAISS) ---")
for c in candidatos_faiss:
    print(f"- {c['texto']} (Vigente: {c['vigente']})")

# Aplicamos una regla de negocio heurística: Penalizamos/movemos al final los documentos no vigentes
candidatos_reordenados = sorted(candidatos_faiss, key=lambda x: x['vigente'], reverse=True)

print("\n--- RESULTADOS LUEGO DEL RE-RANKING (Priorizando Vigencia) ---")
for c in candidatos_reordenados:
    print(f"- {c['texto']}")

# Generación Final con el mejor candidato re-rankeado
mejor_contexto = candidatos_reordenados[0]['texto']
prompt_final = f"Responde formalmente a la pregunta basada estrictamente en este contexto vigente.\nContexto: {mejor_contexto}\nPregunta: {query_optimizada}"

print("\n--- GENERACIÓN FINAL GROUNDED ---")
print(client.models.generate_content(contents=prompt_final, model=MODELO_LLM).text)


--- POST-RETRIEVAL (RESULTADOS CRUDOS DE FAISS) ---
- Reglamento 2022 (Histórico): Se exige un 75% de asistencia mínima para posgrados. (Vigente: False)
- Resolución UBA 2026: La asistencia obligatoria en la Maestría en Ciencias de Datos es del 80%. (Vigente: True)
- Reglamento General: Las llegadas tarde de más de 30 minutos cuentan como media falta. (Vigente: True)

--- RESULTADOS LUEGO DEL RE-RANKING (Priorizando Vigencia) ---
- Resolución UBA 2026: La asistencia obligatoria en la Maestría en Ciencias de Datos es del 80%.
- Reglamento General: Las llegadas tarde de más de 30 minutos cuentan como media falta.
- Reglamento 2022 (Histórico): Se exige un 75% de asistencia mínima para posgrados.

--- GENERACIÓN FINAL GROUNDED ---
En respuesta a su consulta, y basándonos estrictamente en el contexto vigente proporcionado ("Resolución UBA 2026"), se detalla lo siguiente:

**1. Umbral Máximo de Inasistencias Permitidas:**
Según la "Resolución UBA 2026", la asistencia obligatoria en la Maes